# cAPTure: development OOF operational evaluation

This notebook compares XGB-P, both context ablations, and full XGB-P+T under the same predeclared false-alert budgets. It reads only checksum-verified development OOF artifacts. A separate score threshold is selected for each model and budget from development OOF scores; held-out author-train and final-test scenario contents are not read.


## 1. Prepare the Colab environment


In [1]:
from google.colab import drive
drive.mount("/content/drive")

from datetime import datetime, timezone
from pathlib import Path
import subprocess
import sys

REPOSITORY_URL = "https://github.com/tatipar/temporalgnn-nids.git"
REPOSITORY_BRANCH = "feat/capture-feasibility"
PROJECT_ROOT = Path("/content/temporalgnn-nids")
DRIVE_ROOT = Path("/content/drive/MyDrive/capture_gate0")
if not PROJECT_ROOT.exists():
    subprocess.run(["git", "clone", "--branch", REPOSITORY_BRANCH,
                    "--single-branch", REPOSITORY_URL, str(PROJECT_ROOT)], check=True)
branch = subprocess.check_output(["git", "branch", "--show-current"],
                                 cwd=PROJECT_ROOT, text=True).strip()
if branch != REPOSITORY_BRANCH:
    raise RuntimeError(f"Expected branch {REPOSITORY_BRANCH}, found {branch}.")
required_files = ["code/python/requirements-capture-xgb.txt",
                  "code/python/utils/capture_xgb_p_t.py",
                  "code/python/utils/capture_oof_operational.py",
                  "configs/capture_experiment_v1.yaml"]
missing = [name for name in required_files if not (PROJECT_ROOT / name).is_file()]
if missing:
    raise FileNotFoundError(f"Update the Colab repository copy first: {missing}")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r",
                str(PROJECT_ROOT / "code/python/requirements-capture-xgb.txt")], check=True)
sys.path.insert(0, str(PROJECT_ROOT / "code/python"))
import pandas as pd
from IPython.display import display
print("Repository commit:", subprocess.check_output(
    ["git", "rev-parse", "HEAD"], cwd=PROJECT_ROOT, text=True).strip())


Mounted at /content/drive
Repository commit: 5f7a771e66d5d36eda0e3383965ca63d0d2ed8b4


## 2. Bind the completed model runs

Enter the completed XGB-P+T and ablation run IDs. Set `EVALUATION_RUN_ID` only when resuming an existing evaluation. The evaluator validates fold assignments, artifact checksums, packet counts, context provenance, and the frozen manifest policy before reporting metrics.


In [3]:
from utils.capture_oof_operational import (
    run_operational_oof_evaluation, validate_operational_run,
)

MANIFEST_PATH = PROJECT_ROOT / "configs/capture_experiment_v1.yaml"
BASELINE_RUN_DIR = DRIVE_ROOT / "xgb_p_runs/20260919T151844_852400Z_xgb_p"
PRIMARY_RUN_ID = "20260919T201909_754628Z_xgb_p_t"
ABLATION_RUN_ID = "20260920T000745_916321Z_xgb_p_t_ablation"
EVALUATION_RUN_ID = None  # Set only when resuming an existing evaluation.
if PRIMARY_RUN_ID is None or ABLATION_RUN_ID is None:
    raise ValueError("Set PRIMARY_RUN_ID and ABLATION_RUN_ID before evaluating OOF runs.")
if EVALUATION_RUN_ID is None:
    EVALUATION_RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ") + "_operational_oof"
ABLATION_DIR = DRIVE_ROOT / "xgb_p_t_ablation_runs" / ABLATION_RUN_ID
RUN_DIRS = {
    "xgb_p": BASELINE_RUN_DIR,
    "current_window": ABLATION_DIR / "current_window",
    "history": ABLATION_DIR / "history",
    "full": DRIVE_ROOT / "xgb_p_t_runs" / PRIMARY_RUN_ID,
}
OUTPUT_DIR = DRIVE_ROOT / "operational_oof_runs" / EVALUATION_RUN_ID
print("Input runs:", RUN_DIRS)
print("Evaluation output:", OUTPUT_DIR)


Input runs: {'xgb_p': PosixPath('/content/drive/MyDrive/capture_gate0/xgb_p_runs/20260919T151844_852400Z_xgb_p'), 'current_window': PosixPath('/content/drive/MyDrive/capture_gate0/xgb_p_t_ablation_runs/20260920T000745_916321Z_xgb_p_t_ablation/current_window'), 'history': PosixPath('/content/drive/MyDrive/capture_gate0/xgb_p_t_ablation_runs/20260920T000745_916321Z_xgb_p_t_ablation/history'), 'full': PosixPath('/content/drive/MyDrive/capture_gate0/xgb_p_t_runs/20260919T201909_754628Z_xgb_p_t')}
Evaluation output: /content/drive/MyDrive/capture_gate0/operational_oof_runs/20260920T142514_611048Z_operational_oof


## 3. Compute or verify the immutable development report

The primary budget is one false-alert window per hour. Sensitivity budgets are one per 12 hours and one per five minutes. A window alerts when its maximum packet score is at least the model threshold. Only windows without malicious packets contribute false alerts and benign exposure; empty wall-clock windows contribute exposure but cannot alert. The threshold is the most permissive one whose worst fold mean scenario rate meets the budget. One attack-step iteration is detected only when one of its malicious packets crosses the threshold, at that packet window's close.


In [4]:
if OUTPUT_DIR.exists():
    report = validate_operational_run(OUTPUT_DIR, MANIFEST_PATH, RUN_DIRS)
else:
    report = run_operational_oof_evaluation(
        manifest_path=MANIFEST_PATH, run_dirs=RUN_DIRS,
        output_dir=OUTPUT_DIR, batch_size=250_000)
print("Evaluation run ID:", EVALUATION_RUN_ID)
print("Budgets per hour:", report["budgets_per_hour"])


Aggregating xgb_p OOF windows and iterations...
Aggregating current_window OOF windows and iterations...
Aggregating history OOF windows and iterations...
Aggregating full OOF windows and iterations...
Evaluation run ID: 20260920T142514_611048Z_operational_oof
Budgets per hour: {'one_per_hour': 1.0, 'one_per_12_hours': 0.08333333333333333, 'one_per_5_minutes': 12.0}


## 4. Review thresholds and primary-budget metrics

Compare sequence detection rate and latency alongside false alerts. The detected-only latency is diagnostic; the main latency summary retains missed iterations using their attack-step duration. Review individual scenarios before interpreting the hierarchical macro result.


In [5]:
budget_name = "one_per_hour"
threshold_rows = []
macro_rows = []
scenario_rows = []
for model_name, model_report in report["models"].items():
    selected = model_report["thresholds"][budget_name]
    threshold_rows.append({
        "model": model_name,
        "score_threshold": selected["threshold"],
        "worst_fold_false_alert_windows_per_hour": selected[
            "worst_fold_false_alert_windows_per_hour"],
    })
    result = model_report["budgets"][budget_name]
    macro_rows.append({"model": model_name, **result["hierarchical_macro"]})
    for scenario, metrics in result["scenario_metrics"].items():
        scenario_rows.append({"model": model_name, "scenario": scenario, **metrics})
display(pd.DataFrame(threshold_rows).set_index("model"))
display(pd.DataFrame(macro_rows).set_index("model"))
display(pd.DataFrame(scenario_rows).set_index(["model", "scenario"])[[
    "fold", "false_alert_windows_per_hour", "packet_recall",
    "sequence_detection_rate", "mean_miss_capped_latency_seconds",
    "attack_step_iterations", "detected_iterations"]])


,score_threshold,worst_fold_false_alert_windows_per_hour
model,,
xgb_p,0.999463,0.417543
current_window,0.983387,0.798887
history,0.995854,0.979633
full,0.997506,0.866198


,false_alert_windows_per_hour,packet_recall,packet_false_positive_rate,sequence_detection_rate,mean_miss_capped_latency_seconds
model,,,,,
xgb_p,0.208772,0.600363,0.000030,0.528501,63.809228
current_window,0.658179,0.827336,0.000128,0.909566,2.760956
history,0.918791,0.815902,0.000027,0.985104,3.081228
full,0.433099,0.771486,0.000068,0.844987,5.912042


fold  false_alert_windows_per_hour  \
model          scenario                                               
xgb_p          train_dollar_char    A                      0.495441   
               train_slash_char     A                      0.325792   
               train_sub_exf        A                      0.431396   
               train_empty_conn     B                      0.000000   
               train_qos_mid        B                      0.000000   
current_window train_dollar_char    A                      0.990883   
               train_slash_char     A                      0.542986   
               train_sub_exf        A                      0.862792   
               train_empty_conn     B                      0.707965   
               train_qos_mid        B                      0.326975   
history        train_dollar_char    A                      0.990883   
               train_slash_char     A                      0.977376   
               train_sub_exf        A                      0.970641   
               train_empty_conn     B                      1.061947   
               train_qos_mid        B                      0.653951   
full           train_dollar_char    A                      0.867022   
               train_slash_char     A                      0.868778   
               train_sub_exf        A                      0.862792   
               train_empty_conn     B                      0.000000   
               train_qos_mid        B                      0.000000   

                                  packet_recall  sequence_detection_rate  \
model          scenario                                                    
xgb_p          train_dollar_char       0.327993                 0.741379   
               train_slash_char        0.883428                 0.828652   
               train_sub_exf           0.536327                 0.570149   
               train_empty_conn        0.645770                 0.429907   
               train_qos_mid           0.590517                 0.257310   
current_window train_dollar_char       0.640818                 0.974138   
               train_slash_char        0.926638                 0.997191   
               train_sub_exf           0.940674                 0.710448   
               train_empty_conn        0.904870                 0.897196   
               train_qos_mid           0.732384                 0.953216   
history        train_dollar_char       0.613390                 1.000000   
               train_slash_char        0.949800                 0.997191   
               train_sub_exf           0.892591                 0.913433   
               train_empty_conn        0.876265                 1.000000   
               train_qos_mid           0.750156                 1.000000   
full           train_dollar_char       0.520783                 0.948276   
               train_slash_char        0.926562                 0.988764   
               train_sub_exf           0.920410                 0.614925   
               train_empty_conn        0.884988                 0.766355   
               train_qos_mid           0.622452                 0.912281   

                                  mean_miss_capped_latency_seconds  \
model          scenario                                              
xgb_p          train_dollar_char                         31.859447   
               train_slash_char                          22.649773   
               train_sub_exf                             33.582168   
               train_empty_conn                          71.641746   
               train_qos_mid                            124.867575   
current_window train_dollar_char                          3.258431   
               train_slash_char                           2.626940   
               train_sub_exf                              2.449528   
               train_empty_conn                           2.717260   
               train_qos_mi

## 5. Review budget sensitivity and weak attack steps

The two sensitivity budgets use thresholds selected by the same rule. Missed iterations remain in the saved report for independent review.


In [6]:
sensitivity_rows = []
for model_name, model_report in report["models"].items():
    for budget_name in report["budget_order"]:
        selected = model_report["thresholds"][budget_name]
        summary = model_report["budgets"][budget_name]["hierarchical_macro"]
        sensitivity_rows.append({"model": model_name, "budget": budget_name,
                                 "threshold": selected["threshold"], **summary})
display(pd.DataFrame(sensitivity_rows).set_index(["model", "budget"]))
step_rows = []
for model_name, model_report in report["models"].items():
    for step in model_report["budgets"]["one_per_hour"]["step_metrics"].values():
        step_rows.append({"model": model_name, **step})
display(pd.DataFrame(step_rows).sort_values(
    ["sequence_detection_rate", "mean_miss_capped_latency_seconds"],
    ascending=[True, False]).head(40))


threshold  false_alert_windows_per_hour  \
model          budget                                                       
xgb_p          one_per_hour        0.999463                      0.208772   
               one_per_12_hours    0.999942                      0.000000   
               one_per_5_minutes   0.987051                      3.163539   
current_window one_per_hour        0.983387                      0.658179   
               one_per_12_hours    0.994354                      0.000000   
               one_per_5_minutes   0.959686                      5.655613   
history        one_per_hour        0.995854                      0.918791   
               one_per_12_hours    0.999206                      0.036074   
               one_per_5_minutes   0.988180                      6.718528   
full           one_per_hour        0.997506                      0.433099   
               one_per_12_hours    0.998009                      0.036074   
               one_per_5_minutes   0.996420                      4.158372   

                                  packet_recall  packet_false_positive_rate  \
model          budget                                                         
xgb_p          one_per_hour            0.600363                3.025554e-05   
               one_per_12_hours        0.518892                2.932546e-07   
               one_per_5_minutes       0.733198                1.547785e-04   
current_window one_per_hour            0.827336                1.282779e-04   
               one_per_12_hours        0.794438                1.991183e-05   
               one_per_5_minutes       0.846149                5.335559e-04   
history        one_per_hour            0.815902                2.720009e-05   
               one_per_12_hours        0.721846                1.158276e-06   
               one_per_5_minutes       0.869324                2.594214e-04   
full           one_per_hour            0.771486                6.803507e-05   
               one_per_12_hours        0.766824                1.104592e-05   
               one_per_5_minutes       0.780297                1.826829e-04   

                                  sequence_detection_rate  \
model          budget                                       
xgb_p          one_per_hour                      0.528501   
               one_per_12_hours                  0.395452   
               one_per_5_minutes                 0.994527   
current_window one_per_hour                      0.909566   
               one_per_12_hours                  0.838935   
               one_per_5_minutes                 0.974778   
history        one_per_hour                      0.985104   
               one_per_12_hours                  0.860758   
               one_per_5_minutes                 0.995522   
full           one_per_hour                      0.844987   
               one_per_12_hours                  0.827461   
               one_per_5_minutes                 0.863420   

                                  mean_miss_capped_latency_seconds  
model          budget                                               
xgb_p          one_per_hour                              63.809228  
               one_per_12_hours                          70.745783  
               one_per_5_minutes                          2.584426  
current_window one_per_hour                               2.760956  
               one_per_12_hours                           4.838608  
               one_per_5_minutes                          2.678593  
history        one_per_hour                               3.081228  
               one_per_12_hours                           9.643856  
               one_per_5_minutes                          2.563735  
full           one_per_hour                               5.912042  
               one_per_12_hours                           6.515564  
               one_per_5_minutes                          4.334954

,model,scenario,attack_step,iterations,detected_iterations,sequence_detection_rate,mean_miss_capped_latency_seconds
25,xgb_p,train_empty_conn,empty_conn,37,0,0.000000,94.925403
39,xgb_p,train_qos_mid,qos_mid,39,0,0.000000,20.590014
18,xgb_p,train_sub_exf,mqtt_cat,27,0,0.000000,5.365242
27,xgb_p,train_empty_conn,mqtt_cat,12,0,0.000000,5.302040
34,xgb_p,train_qos_mid,mqtt_cat,10,0,0.000000,5.222620
41,xgb_p,train_qos_mid,scp_inst,5,0,0.000000,0.199340
133,full,train_dollar_char,scp_inst,7,0,0.000000,0.192010
23,xgb_p,train_sub_exf,scp_exf,117,0,0.000000,0.184702
15,xgb_p,train_slash_char,sftp_inst,1,0,0.000000,0.166788
141,full,train_slash_char,sftp_inst,1,0,0.000000,0.166788
